**Building an RAG Pipeline Using PDF Chunking and Retrieval**

Build a a Retrieval-Augmented-Generation Pipeline using various tools in langchain library to process PDF documents , convert them into vectorized chunks and retrieve relevant information using semantic search .
Learnings  - STrengthen understanding on-  real world content


*   Document chunking
*   Embedding
*   Vector search workflow


Relative path for loading the pdfs-
<!--
/content/sample_data/rag_pdfs/beyond-the-copilot-scaling-the-agentic-product-development-life-cycle.pdf

/content/sample_data/rag_pdfs/one-year-of-agentic-ai-six-lessons-from-the-people-doing-the-work_vf.pdf -->

/home/subhroy557/ai-rag_agent/cep_data_monetization_problem_2.pdf
/home/subhroy557/ai-rag_agent/cep_data_monetization_problem_11.pdf

# Step 1: Install dependencies including pdf and text loaders

In [1]:
# Import the libraries , all of the tools for agent building
!pip install langchain openai PyPDF2 faiss-cpu tiktoken langchain-openai langchain-classic pypdf
!pip install langchain_community
!pip install langchain_text_splitters
!pip install langchain_classic

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 kB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.2/110.2 kB 4.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.9/45.9 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.5/161.5 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 17.6 MB/s eta 0:00:00m eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 18.3 MB/s eta 0:00:00m eta 0:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 19.0 MB/s eta 0:00:000m eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.8/125.8 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 24.9 MB/s eta 0:00:000m et

# Step 2: Import dependencies

In [2]:
import os
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader
from langchain_openai import AzureChatOpenAI, AzureOpenAIEmbeddings
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter # for chunking
# vectorstores is for storage purposes  # computers only understand numbers hence vectorize the text, called embeddings store in vector db
# FAISS is one of the classes helping to store the embeddings in a vector db
from langchain_classic.chains import RetrievalQA  # helping to load some documets , load some embeddings
#update

from langchain_community.vectorstores import FAISS # This is for
from langchain_community.document_loaders import TextLoader
# import the libraries for PDFs
from langchain_community.document_loaders import PyPDFLoader
from PyPDF2 import PdfReader

/tmp/ipykernel_5352/1889451621.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader


# Step 3: Set Azure OpenAI credentials

In [3]:
os.environ["AZURE_OPENAI_API_KEY"] = "2ABecnfxzhRg4M5D6pBKiqxXVhmGB2WvQ0aYKkbTCPsj0JLKsZPfJQQJ99BDAC77bzfXJ3w3AAABACOGi3sC"# same clone of the lab sessions

# Step 4: Load and chunk your custom educational documents

Preserves semantic boundaries, making chunks safer for embeddings, search, and RAG workflows



In [4]:
from langchain_core import documents
loader = DirectoryLoader(
    path ='/home/subhroy557/ai-rag_agent',
    #ath='/content/sample_data/',
    glob='*.pdf',
    loader_cls=PyPDFLoader
)

documents = loader.lazy_load() # for performance

for document in documents:
    print(document.metadata)

{'producer': 'Skia/PDF m154 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'cep_data_monetization_problem_2.docx', 'source': '/home/subhroy557/ai-rag_agent/cep_data_monetization_problem_2.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1'}
{'producer': 'Skia/PDF m154 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'cep_data_monetization_problem_2.docx', 'source': '/home/subhroy557/ai-rag_agent/cep_data_monetization_problem_2.pdf', 'total_pages': 4, 'page': 1, 'page_label': '2'}
{'producer': 'Skia/PDF m154 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'cep_data_monetization_problem_2.docx', 'source': '/home/subhroy557/ai-rag_agent/cep_data_monetization_problem_2.pdf', 'total_pages': 4, 'page': 2, 'page_label': '3'}
{'producer': 'Skia/PDF m154 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'cep_data_monetization_problem_2.docx', 'source': '/home/subhroy557/ai-rag_agent/cep_data_monetizat

Step 5: Split into chucks all of the loaded documents

In [5]:
#LangChain's recommended default strategy for breaking large documents into smaller, context-preserving chunks , moves on to strategy in a top down hierarchical manner
# such as paragraphs , newline , blank spaces and empty srings
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)# how do we define the chunk size ? overlap is needed to retain context across chunks

# Re-load the documents because the generator was consumed in the previous cell.
documents = loader.lazy_load()
# Convert the documents generator to a list.
documents_list = list(documents)

docs = text_splitter.split_documents(documents_list)

# Check if the loader is initialized correctly
print(f'Loaded {len(documents_list)} documents')
print(f'Split into {len(docs)} chunks')

Loaded 11 documents
Split into 36 chunks


# Step 5: Create vectorstore using Azure embeddings
---
## Some facts about FAISS
FAISS is a library for efficient similarity search and clustering of dense vectors. It is designed to handle datasets ranging from a few million to over a billion high-dimensional vectors, making it a backbone for modern recommendation systems, search engines, and AI applications. FAISS provides highly optimized algorithms and data structures for nearest neighbor search (KNN) and clustering, using both CPUs and GPUs for maximum performance.

In [6]:
# Embed documnet chunks and store them in an FAISS vector store
embedddings = AzureOpenAIEmbeddings(
    azure_endpoint="https://openai-api-management-gw.azure-api.net",
    deployment="text-embedding-ada-002",
    chunk_size=500,
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    api_version="2023-05-15",
    model="text-embedding-ada-002",
    
)
vectorstore = FAISS.from_documents(docs, embedddings) # Facebook AI Semantic Similarity

# Step 6: Initialize the Azure OpenAI LLM

In [7]:
# init the azure openai llm for deterministic output , set temperature = 0
llm = AzureChatOpenAI(
    azure_endpoint="https://openai-api-management-gw.azure-api.net",
    api_version="2025-01-01-preview",
    deployment_name="gpt-5-mini"
)

\# Step 7: Create the RAG chain

In [8]:
qa_chain= RetrievalQA.from_chain_type(
    llm=llm,
    retriever=vectorstore.as_retriever(),
    return_source_documents=True
)

# Step 8: Ask a question

In [9]:
# Do a semantic search test for a sample query to ensure that the RAG is ableto fetch the relevant content 
# Query for a unique phrase or topic from your PDF 
query = "What relevant datasets are mentioned for high valued customers inside the PDF?"
docs = vectorstore.similarity_search(query, k=2)

for doc in docs:
    print(f"Source: {doc.metadata.get('source')}")
    print(f"Content Snippet: {doc.page_content[:200]}\n")

Source: /home/subhroy557/ai-rag_agent/cep_data_monetization_problem_2.pdf
Content Snippet: 4.  Customer  Segmentation  and  High-Net-Worth  Individual  (HNWI)  Identification  
Relevant  Datasets  
●
 
Bank
 
account
 
balances
 
and
 
transaction
 
patterns
 
●
 
Investment
 
portfolios
 


Source: /home/subhroy557/ai-rag_agent/cep_data_monetization_problem_2.pdf
Content Snippet: enhancing
 
customer
 
satisfaction
 
without
 
directly
 
monetizing
 
the
 
data.
 
3.  Credit  Risk  and  Creditworthiness  Assessment  
Relevant  Datasets  
●
 
Credit
 
card
 
transaction
 
histo



# Step 9: Print results

In [ ]:
# Define a function for clearn results 
def clean_result(result):
    if isinstance(result, dict):
        result = result.get("result", result)
    text = str(result).replace("\\n", "\n").replace("\\t", "\t").strip()
    return text

In [11]:
query1 = "What examples are existing in the documents with respect to Data monetization, provide them in a list?"
result1 = qa_chain.invoke({"query": query1})
print("Result 1:\n" + clean_result(result1))

Result 1:
Here are the data‑monetization examples and example approaches that appear in the provided documents:

- Direct monetization approaches (examples of what “direct” can mean)
  - Selling data externally
  - Licensing data to third parties
  - Exchanging data for payment or reciprocal access

- Indirect monetization (examples of internal use cases / objectives)
  - Improve operational efficiency
  - Reduce risk
  - Enhance decision‑making
  - Enable new internal capabilities

- Named indirect approach with a concrete example
  - Operational optimization — Airlines using weather and operational data to predict flight delays, proactively rebook passengers, reduce refunds, and minimize overtime costs

- Explicit outcomes / uses mentioned that illustrate monetization value
  - Generating explicit revenue streams (direct)
  - Enhancing internal capabilities, efficiency, and decision‑making (indirect)
  - Improved strategic decision‑making and more customer‑centric business models

If

### Detailed Results with Source Documents

Let's print the full results for the query including the `source_documents` that the RAG chain used to formulate its answers. This will clarify what information was retrieved from  PDFs.

In [20]:
print("Source Documents:")
for doc in result1.get("source_documents", []):
    print(f"  - Source: {doc.metadata['source']}, Page: {doc.metadata['page']}")
    print(f"    Content: {doc.page_content[:200]}...") # Print first 200 chars of content
if not result1.get("source_documents"): # Handles case where source_documents might be empty or missing
    print("  No source documents found for this query.")

Source Documents:
  - Source: /home/subhroy557/ai-rag_agent/cep_data_monetization_problem_11.pdf, Page: 0
    Content: Course
 
End
 
Project
 
:
 
Data
 
Monetization
 Question  Part  1
 
For
 
the
 
first
 
part
 
of
 
this
 
exercise,
 
describe
 
the
 
different
 
types
 
of
 
data
 
categories
 
and
 
their
 
def...
  - Source: /home/subhroy557/ai-rag_agent/cep_data_monetization_problem_2.pdf, Page: 0
    Content: Question  Part  2   Within  the  indirect  monetization  category,  select  4  approaches  or  channels  to  illustrate  the  
datasets
 
you
 
have
 
within
 
your
 
organization
 
that
 
could
 
pla...
  - Source: /home/subhroy557/ai-rag_agent/cep_data_monetization_problem_11.pdf, Page: 5
    Content: monetization
 
focuses
 
on
 
using
 
data
 
internally
 
to
 
improve
 
efficiency,
 
reduce
 
risk,
 
enhance
 
decision-making,
 
or
 
enable
 
new
 
capabilities
,
 
rather
 
than
 
selling
 
data...
  - Source: /home/subhroy557/ai-rag_agent/cep_data_monetization_prob

In [12]:
# Print the most relvant chunk ie the first chunk by ranking 
# The list of retrieved source document chunks
source_chunks = result1["source_documents"]

# Get the most relevant (first) chunk for result1 
most_relevant_chunk = source_chunks[0]
print("Most relevant chunk for result1",most_relevant_chunk.page_content)


Most relevant chunk for result1 Course
 
End
 
Project
 
:
 
Data
 
Monetization
 Question  Part  1
 
For
 
the
 
first
 
part
 
of
 
this
 
exercise,
 
describe
 
the
 
different
 
types
 
of
 
data
 
categories
 
and
 
their
 
definitions
 
in
 
terms
 
of
 
transformational
 
value.
 
For
 
each
 
of
 
the
 
categories,
 
expand
 
on
 
at
 
least
 
3
 
examples
 
of
 
approaches
 
or
 
channels
 
to
 
execute
 
this,
 
giving
 
of
 
these
 
approaches
 
or
 
channels.
 
Lastly,
 
for
 
the
 
direct
 
sales
 
monetization


In [17]:
query2 = "What are the different types of Data monetization described in the documents?"
result2 = qa_chain.invoke({"query": query2})

print("Result 2:\n" + clean_result(result2))

Result 2:
The documents describe two broad types of data monetization:

1. Direct data monetization
- Definition: selling, licensing or exchanging data or data-driven products externally to generate immediate, measurable revenue.
- Example form noted in the documents: direct sale of raw, anonymized, aggregated or insight-based data products.

2. Indirect data monetization
- Definition: deriving value from data by enhancing internal capabilities, efficiency, decision‑making or product/service offerings—value realized through improved operations, better products, reduced costs or increased customer lifetime value rather than an immediate external revenue stream.


In [14]:
query3 = "What do the customers of Medtronic gain as per the example of Data monetization?"
result3 = qa_chain.invoke({"query": query3})
print("Result 3:\n" + clean_result(result3))

Result 3:
They gain detailed, integrated reports that help them optimize therapy, make data‑driven lifestyle recommendations, and improve patient outcomes.


In [19]:
# Retrieve and print the most relevant chunk for result3 
# Print the most relvant chunk ie the first chunk by ranking 
# The list of retrieved source document chunks
source_chunks = result3["source_documents"]

# Get the most relevant (first) chunk for result1 
most_relevant_chunk = source_chunks[0]
print("Most relevant chunk for result3",most_relevant_chunk.page_content)

Most relevant chunk for result3 ●
 
Fitbit:
 
Expands
 
from
 
consumer
 
health
 
tracking
 
into
 
clinically
 
validated
 
medical
 
use
 
cases,
 
strengthening
 
brand
 
credibility
 
and
 
increasing
 
potential
 
sales.
 
 
●
 
Patients:
 
Benefit
 
from
 
improved
 
health
 
outcomes,
 
reduced
 
GP
 
visits,
 
and
 
potentially
 
lower
 
health
 
insurance
 
costs
 
due
 
to
 
improved
 
patient
 
health.
 
 
2.  Indirect  Data  monetization   
Definition:
 
 
Indirect
 
monetization
 
focuses
 
on
 
using
 
data


Step 10: Calculate the total relevant chunks retrived and the average chunk size 

In [18]:
# Step 10: Calculate total relevant chunks retrieved and average chunk size captured across all queries and results 
retrieved_chunks = []

for result in [result1, result2, result3]:
    retrieved_chunks.extend(result.get("source_documents", []))

# Total number of chunks returned as relevant across all queries
total_relevant_chunks = len(retrieved_chunks)

# Average chunk size using character count (and word count as a useful secondary metric)
chunk_sizes_chars = [len(doc.page_content) for doc in retrieved_chunks]
chunk_sizes_words = [len(doc.page_content.split()) for doc in retrieved_chunks]

average_chunk_size_chars = sum(chunk_sizes_chars) / len(chunk_sizes_chars) if chunk_sizes_chars else 0
average_chunk_size_words = sum(chunk_sizes_words) / len(chunk_sizes_words) if chunk_sizes_words else 0

print(f"Total relevant chunks retrieved: {total_relevant_chunks}")
print(f"Average chunk size: {average_chunk_size_chars:.2f} characters ({average_chunk_size_words:.2f} words)")

Total relevant chunks retrieved: 12
Average chunk size: 448.50 characters (49.42 words)
